# WorldSplat-Prune - post-reconstruction pruning (spec 4-5)

Drives the pruning operator end to end on a trained model: lift labels (§3,
`Test_Vote.ipynb`) → plan the arms (§4-5) → visualise → write PLYs → score with
`eval_worldsplat`.

| stage | module | output |
|---|---|---|
| 1 | `vote.vote_classes_over_views` | `VoteBundle` (seg, purity, boundary_dist, ...) |
| 2 | `prune.plan_pruning` | keep/delete per arm + count-contract report |
| 3 | `scene.gaussian_model` | pruned PLYs (`pruned_<arm>_k<frac>/`) |
| 4 | `eval_worldsplat` | PSNR / SSIM / LPIPS, band + interior PSNR, count |

The **arms** are the control matrix (§5.2): `semantic` (band + purity),
`geometry` (the control: neither), `random` (structure bar). All are matched to
the same Gaussian count via `KEEP_FRAC`.

> Needs a trained PLY, the CUDA rasterizer and labels - same prerequisites as
> `Test_Vote.ipynb` (section 2 builds the deps).


## 1. Mount Google Drive

Skip if you run locally.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print("not Colab / already mounted:", e)

## 2. Get this repo + build the rasterizer

Clones the deps directly (the superproject records no submodule gitlinks) and
pins the rasterizer to the fork's `dr_aa` branch; populates the vendored `glm`.


In [ ]:
import os, subprocess, shutil

REPO_URL = "https://github.com/satyaba/worldsplat-prune.git"
REPO_DIR = "/content/worldsplat-prune"
BRANCH = "main"
RASTERIZER_URL = "https://github.com/graphdeco-inria/diff-gaussian-rasterization.git"
KNN_URL = "https://gitlab.inria.fr/bkerbl/simple-knn.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH, "-q"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)

os.chdir(REPO_DIR)


def _git(args):
    subprocess.run(["git"] + args, check=True)


def ensure_clone(path, url, ref=None):
    if not os.path.exists(os.path.join(path, ".git")):
        shutil.rmtree(path, ignore_errors=True)
        _git(["clone", "-q", url, path])
    if ref is not None:
        _git(["-C", path, "fetch", "-q", "origin", ref])
        _git(["-C", path, "checkout", "-q", "FETCH_HEAD"])
    _git(["-C", path, "submodule", "update", "--init", "--recursive"])


ensure_clone("submodules/diff-gaussian-rasterization", RASTERIZER_URL, ref="dr_aa")
ensure_clone("submodules/simple-knn", KNN_URL)

!bash scripts/patch_cuda_submodules.sh
!pip install -q plyfile opencv-python joblib
!pip install -q ./submodules/diff-gaussian-rasterization --no-build-isolation
!pip install -q ./submodules/simple-knn --no-build-isolation

print("cwd:", os.getcwd())

## 3. GPU check


In [ ]:
!nvidia-smi

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configure paths + arms

`MODEL_DIR = $HOME_DIR/output/<MODEL_NAME>` (the trained model). `KEEP_FRAC` is
the matched budget across arms; `BAND`/`STRENGTH`/`CURVE` are the §4-5
hyperparameters ([D1]/[D12]); the vote knobs mirror `Test_Vote.ipynb`.


In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"
MODEL_NAME = "base_30k"
ITERATION = -1

os.environ["HOME_DIR"] = HOME_DIR
DATASET_ROOT = Path(HOME_DIR) / "Dataset"
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)

CANDIDATES = [
    DATASET_ROOT / DATASET_NAME,
    Path(HOME_DIR) / "raw" / Path(DATASET_NAME).name,
    Path(DATASET_NAME),
]
DATASET_DIR = next((c for c in CANDIDATES if (c / "images").is_dir()), None)
MODEL_DIR = Path(HOME_DIR) / "output" / MODEL_NAME
OUT_ROOT = Path(HOME_DIR) / "output"
assert DATASET_DIR is not None, "no 'images/' dir found - edit DATASET_NAME"
assert MODEL_DIR.exists(), f"trained model not found: {MODEL_DIR}"

# --- vote pass (§3) --------------------------------------------------------
LABEL_ADAPTER = "composite"
LABEL_DIR = None
MIN_AREA = 0
VISIBILITY = "render"
VOTE = "first"
DT_NORM = 100.0
MIN_VIEWS = 3
OCCLUSION = True
DEPTH_TOL = 0.02
MASK_MARGIN = 2.0

# --- pruning (§4-5) --------------------------------------------------------
KEEP_FRAC = 0.5          # matched budget for every arm
BAND = 0.04              # d_i < BAND protects (== 4 px at DT_NORM=100)
EPSILON = 0.005          # alpha floor for the dead set
STRENGTH = 0.5           # sigma: semantic strength
CURVE = 1.0              # a: purity curve
BOUNDARY_BAND_PX = 4     # eval band width (px)

# control matrix (§5.2): each entry is the plan_pruning kwargs for an arm
ARMS = {
    "semantic": dict(use_band=True,  use_purity=True),
    "geometry": dict(use_band=False, use_purity=False),   # control
    "random":   dict(use_band=False, random_control=True), # structure bar: random, no band
}
print("model :", MODEL_DIR)
print("scene :", DATASET_DIR)
print("arms  :", list(ARMS), "| keep_frac:", KEEP_FRAC)

## 5. Load the trained model + scene


In [ ]:
from argparse import ArgumentParser, Namespace
from pathlib import Path

from arguments import ModelParams
from eval_worldsplat import _fill_model_defaults
from scene import Scene
from scene.gaussian_model import GaussianModel

_parser = ArgumentParser()
_mp = ModelParams(_parser, sentinel=True)
cfg = vars(_parser.parse_args([]))
_cfg_path = Path(MODEL_DIR) / "cfg_args"
if _cfg_path.exists():
    cfg.update(vars(eval(_cfg_path.read_text())))
cfg["source_path"] = str(DATASET_DIR)
cfg["model_path"] = str(MODEL_DIR)
dataset = _mp.extract(Namespace(**cfg))
dataset.depths = ""
_fill_model_defaults(dataset)

gaussians = GaussianModel(dataset.sh_degree)
scene = Scene(dataset, gaussians, load_iteration=ITERATION, shuffle=False)
cams = scene.getTrainCameras()
print("gaussians:", gaussians.get_xyz.shape[0],
      "| train cameras:", len(cams),
      "| iteration:", scene.loaded_iter,
      "| extent: %.3f" % scene.cameras_extent)

## 6. Semantic store


In [ ]:
from worldsplat_prune.datasets import get_adapter
from worldsplat_prune.semantic_store import SemanticStore

adapter = get_adapter(LABEL_ADAPTER, DATASET_DIR,
                      {"label_dir": LABEL_DIR, "min_area": MIN_AREA})
store = SemanticStore(adapter, device="cuda" if torch.cuda.is_available() else "cpu")
print(adapter.meta())

## 7. Vote pass (label lifting)


In [ ]:
from worldsplat_prune.vote import vote_classes_over_views

bundle = vote_classes_over_views(
    gaussians, cams, store,
    visibility=VISIBILITY, vote=VOTE, dt_norm=DT_NORM, min_views=MIN_VIEWS,
    occlusion=OCCLUSION, depth_tol=DEPTH_TOL, mask_margin=MASK_MARGIN,
    camera_order="sorted",
)
print(bundle.summary())
print("dropped: occluded =", bundle.meta["n_occluded"],
      "| mask rim =", bundle.meta["n_masked"])

## 8. Plan the arms (§4-5)

Each arm turns the geometry (`alpha`, `scaling`) + the `VoteBundle` into a
keep/delete decision at the same `KEEP_FRAC`. The `semantic` vs `geometry`
**Jaccard** of their kept sets is the §6 diagnostic: if it is ~1.0, the label
signal barely moves the partition and the arm will be inert regardless of score
quality.


In [ ]:
import numpy as np
from worldsplat_prune import prune as prune_mod
from worldsplat_prune.prune import plan_pruning, jaccard

alpha = gaussians.get_opacity
scaling = gaussians.get_scaling
N0 = int(gaussians.get_xyz.shape[0])
print(f"original gaussians: {N0} | target keep: {int(round(KEEP_FRAC*N0))}")

plans = {}
for name, kw in ARMS.items():
    plans[name] = plan_pruning(
        alpha, scaling,
        boundary_dist=bundle.boundary_dist, seg=bundle.seg, purity=bundle.purity,
        extent=scene.cameras_extent,
        keep_frac=KEEP_FRAC, band=BAND, epsilon=EPSILON,
        strength=STRENGTH, curve=CURVE, **kw)

header = f"{'arm':<10} {'kept':>8} {'k_dead':>7} {'k_cont':>7} {'cont%':>6} {'band':>7} {'floor':>6} {'ceil':>5}"
print(header); print("-" * len(header))
for name, p in plans.items():
    r = p.report()
    print(f"{name:<10} {r['n_after']:>8} {r['k_dead']:>7} {r['k_contested']:>7} "
          f"{100*r['contested_share']:>5.1f}% {r['n_band']:>7} "
          f"{str(r['reachable_floor']):>6} {str(r['reachable_ceiling']):>5}")

print("\nJaccard(semantic, geometry) kept sets:",
      round(jaccard(plans['semantic'].keep, plans['geometry'].keep), 3))
print("(near 1.0 -> the semantic partition barely moves the set => inert arm)")

## 9. Visualise the pruned models

Render a couple of train views from each arm's **kept subset**, plus an overlay
of what the semantic arm deletes (grey = kept, red = deleted) on the full model.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
from argparse import ArgumentParser
from arguments import PipelineParams
from gaussian_renderer import render

_p = ArgumentParser()
pipe = PipelineParams(_p).extract(_p.parse_args([]))
bg = torch.tensor([1, 1, 1] if dataset.white_background else [0, 0, 0],
                  dtype=torch.float32, device="cuda")


def subset_model(src, keep):
    """A new GaussianModel holding only the kept Gaussians (for save/render)."""
    out = GaussianModel(src.max_sh_degree)
    out.active_sh_degree = src.active_sh_degree
    for attr in ("_xyz", "_features_dc", "_features_rest",
                 "_opacity", "_scaling", "_rotation"):
        setattr(out, attr, nn.Parameter(getattr(src, attr)[keep].detach().clone()))
    return out


show = cams[:2]
with torch.no_grad():
    fig, axes = plt.subplots(len(plans), len(show),
                             figsize=(6 * len(show), 4 * len(plans)), squeeze=False)
    for i, (name, p) in enumerate(plans.items()):
        m = subset_model(gaussians, p.keep)
        for j, cam in enumerate(show):
            img = render(cam, m, pipe, bg)["render"].clamp(0, 1)
            axes[i, j].imshow(img.permute(1, 2, 0).cpu())
            axes[i, j].set_title(f"{name} (k={p.n_after})")
            axes[i, j].axis("off")
    plt.tight_layout(); plt.show()

# where the semantic arm deletes (full model, grey kept / red deleted)
p = plans["semantic"]
cols = np.tile(np.array([0.6, 0.6, 0.6], np.float32), (int(p.keep.numel()), 1))
cols[(~p.keep).cpu().numpy()] = [1.0, 0.0, 0.0]
col_t = torch.tensor(cols, dtype=torch.float32, device="cuda")
with torch.no_grad():
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 5), squeeze=False)
    for j, cam in enumerate(show):
        img = render(cam, gaussians, pipe, bg, override_color=col_t)["render"].clamp(0, 1)
        axes[0, j].imshow(img.permute(1, 2, 0).cpu())
        axes[0, j].set_title("semantic deletions (red)")
        axes[0, j].axis("off")
    plt.tight_layout(); plt.show()

## 10. Write the pruned PLYs

Each arm gets a valid model dir (`pruned_<arm>_k<frac>/`) with
`point_cloud/iteration_<n>/point_cloud.ply`, a copied `cfg_args` and the
`prune_report.json`.


In [ ]:
import json, shutil

written = {}
for name, p in plans.items():
    out = OUT_ROOT / f"pruned_{name}_k{KEEP_FRAC:g}"
    pc = out / "point_cloud" / f"iteration_{scene.loaded_iter}"
    pc.mkdir(parents=True, exist_ok=True)
    subset_model(gaussians, p.keep).save_ply(str(pc / "point_cloud.ply"))
    src = Path(MODEL_DIR) / "cfg_args"
    if src.exists():
        shutil.copyfile(src, out / "cfg_args")
    rep = p.report()
    rep["vote"] = {k: bundle.meta[k] for k in
                   ("visibility", "vote", "occlusion", "min_views",
                    "n_labeled", "mean_purity")}
    (out / "prune_report.json").write_text(json.dumps(rep, indent=2))
    written[name] = out
    print(f"{name:<10} -> {out}  ({p.n_after} gaussians)")

## 11. Evaluate the arms

`eval_worldsplat.run` re-renders each pruned model on the **test** split and
reports global + band/interior PSNR (the semantic boundary band comes from the
same label adapter). Comparing `semantic` against the `geometry` control at
matched count is the whole test (§1).


In [ ]:
import copy
import eval_worldsplat as EW

rows = {}
for name, out in written.items():
    ds = copy.copy(dataset)
    ds.model_path = str(out)
    res = EW.run(ds, scene.loaded_iter, pipe,
                 skip_train=True, skip_test=False,
                 boundary_band=BOUNDARY_BAND_PX, adapter_name=LABEL_ADAPTER)
    t = res["splits"]["test"]
    rows[name] = {
        "gaussians": res["gaussians"],
        "psnr": t["psnr"],
        "psnr_boundary": t["psnr_boundary"],
        "psnr_interior": t["psnr_interior"],
        "ssim": t["ssim"],
        "lpips": t["lpips"],
    }

header = f"{'arm':<10} {'count':>8} {'PSNR':>7} {'PSNR_b':>7} {'PSNR_i':>7} {'SSIM':>7} {'LPIPS':>7}"
print(header); print("-" * len(header))
for name, r in rows.items():
    def f(x):
        return "n/a" if x is None else f"{x:.3f}"
    print(f"{name:<10} {r['gaussians']:>8} {f(r['psnr']):>7} {f(r['psnr_boundary']):>7} "
          f"{f(r['psnr_interior']):>7} {f(r['ssim']):>7} {f(r['lpips']):>7}")
print("\nVerdict: semantic beats the geometry control at matched count?",
      "yes" if (rows.get('semantic', {}).get('psnr') or -1) >=
              (rows.get('geometry', {}).get('psnr') or 1e9) else "no / flat")

### Next steps + troubleshooting

- **Sweep the budget:** re-run §8-11 for `KEEP_FRAC in {0.25, 0.5, 0.75}` and
  `STRENGTH in {0.25, 0.5, 0.75}` (§5.1 ablation order: `s`, then `a`, then the
  void/support factors). `experiments/runs.py` will formalise this manifest.
- **Mode (b):** pass `target=<instance_id>` to `plan_pruning` to keep one
  consistent instance whole; judge it by the per-object readout, not the mean.
- **`contested_share` ~ 0:** the run is effectively geometry-only — no semantic
  delta can be attributed (§5.3). Check `prune_report.json`.
- **Jaccard ~ 1.0:** the arm is inert regardless of score — the partition barely
  moved (§6).
- **Band/interior `n/a`:** the label adapter did not resolve for the eval scene;
  confirm `LABEL_ADAPTER`/`DATASET_DIR` match the trained model's `source_path`.
